# 02 — Preprocessing + Patient-Level Split (Phase 3)

Split at the **patient** level (`GroupShuffleSplit`), not the image level — HYGD has an average of 2.6 images per patient, and letting the same patient appear in both train and test would leak information and inflate reported performance.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.data_utils import load_dataset_metadata, train_val_test_split, preprocess_image

df = load_dataset_metadata("../data/raw")
train_df, val_df, test_df = train_val_test_split(df, val_size=0.15, test_size=0.15, seed=42)

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(split_df)} images, {split_df['patient_id'].nunique()} patients, {split_df['label'].mean():.1%} GON+")

**Verified result:** train 535 img / 200 patients (74.8% GON+), val 113 img / 44 patients (73.5% GON+), test 99 img / 44 patients (65.7% GON+). Zero patient overlap between splits (checked directly on the id sets). The test split's GON+ rate is somewhat lower than train/val — `GroupShuffleSplit` balances patient *counts*, not the *label* distribution across groups, and with only 288 patients a small skew is expected. Noted as a known limitation rather than re-engineered away.

In [ ]:
train_p, val_p, test_p = set(train_df.patient_id), set(val_df.patient_id), set(test_df.patient_id)
assert not (train_p & val_p) and not (train_p & test_p) and not (val_p & test_p)
print("No patient overlap across splits — confirmed.")

## Preprocessing

Resize to 224x224 (torchvision ResNet18 input size) and normalize with ImageNet mean/std, since the baseline model reuses ImageNet-pretrained weights. No aggressive augmentation for this baseline — keep it simple per the project's non-goals (see README §2).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = preprocess_image(train_df.iloc[0]["image_path"])
print("preprocessed shape:", sample.shape, 
      "min/max:", sample.min(), sample.max())